# 공공 자전거 악성 미반납 예측
- 목표 : 대여 시점 정보로 24시간 초과 미반납(`is_overdue_24h`) 예측
- 핵심 질문 : "정확도 98%인데 왜 도난을 하나도 못 잡았나?"

## [STEP 1] 파생변수 & 정답지
- `duration_min`, `is_overdue_24h`(1440분 초과 = 1), `hour`, `weekday`
- 미반납 비율, "전부 정상" 예측 시 정확도 확인

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/clean_rentals.csv",parse_dates=["rent_time","return_time"])

print(df.shape)
print(df.dtypes)

(13501, 16)
rental_id                      str
bike_id                        str
user_id                        str
rent_time           datetime64[us]
return_time         datetime64[us]
distance_km                float64
fee                          int64
payment_method                 str
station_id                     str
bike_type                      str
gear_count                 float64
manufacture_year           float64
daily_fee                  float64
station_name                   str
district                       str
duration_min               float64
dtype: object


In [3]:
# 1. 대여시간(분)
diff = df["return_time"] - df["rent_time"]
df["duration_min"] = diff.dt.total_seconds() / 60

print(df[["rent_time", "return_time", "duration_min"]].head())

            rent_time         return_time  duration_min
0 2022-04-18 20:28:00 2022-04-18 20:44:00          16.0
1 2022-05-19 09:53:00 2022-05-19 11:08:00          75.0
2 2022-09-07 16:20:00 2022-09-07 18:04:00         104.0
3 2022-10-11 06:39:00 2022-10-11 07:18:00          39.0
4 2022-07-11 10:15:00 2022-07-11 10:53:00          38.0


In [4]:
# 2. 정답지 : 24시간(1440분) 초과면 1
df["is_overdue_24h"] = (df["duration_min"] > 1440).astype(int)

print(df["is_overdue_24h"].value_counts())

is_overdue_24h
0    13501
Name: count, dtype: int64
